# Algoritmo de Deutsch-Jozsa

## Objetivo
Determinar se uma função é **constante** (retorna sempre 0 ou sempre 1) ou **balanceada** (retorna 0 para metade das entradas e 1 para a outra metade).

## Vantagem Quântica
- **Clássico:** Precisa de até 2^(n-1) + 1 consultas
- **Quântico:** Apenas 1 consulta!

## Referências
- Livro: Capítulo 9
- [Qiskit Textbook](https://qiskit.org/textbook/ch-algorithms/deutsch-jozsa.html)

In [ ]:
# Imports
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

## 1. Criar o Oráculo

Implemente funções que criam oráculos constantes e balanceados.

In [ ]:
def create_constant_oracle(n_qubits: int, output: int = 0) -> QuantumCircuit:
    """
    Cria um oráculo constante.
    
    Args:
        n_qubits: número de qubits de entrada
        output: 0 ou 1 (valor constante retornado)
    
    Returns:
        QuantumCircuit: o oráculo
    """
    # n_qubits de entrada + 1 qubit auxiliar
    oracle = QuantumCircuit(n_qubits + 1)
    
    # Se output = 1, aplicar X no qubit auxiliar
    if output == 1:
        oracle.x(n_qubits)
    
    return oracle


def create_balanced_oracle(n_qubits: int) -> QuantumCircuit:
    """
    Cria um oráculo balanceado.
    
    Args:
        n_qubits: número de qubits de entrada
    
    Returns:
        QuantumCircuit: o oráculo
    """
    oracle = QuantumCircuit(n_qubits + 1)
    
    # Aplicar CNOT de cada qubit de entrada para o auxiliar
    # Isso cria uma função balanceada (XOR de todas as entradas)
    for i in range(n_qubits):
        oracle.cx(i, n_qubits)
    
    return oracle


# Testar os oráculos
print("Oráculo Constante (output=0):")
print(create_constant_oracle(2, 0).draw())
print("\nOráculo Constante (output=1):")
print(create_constant_oracle(2, 1).draw())
print("\nOráculo Balanceado:")
print(create_balanced_oracle(2).draw())

## 2. Implementar o Algoritmo

O circuito segue os passos:
1. Inicializar n qubits em |0⟩ e 1 qubit auxiliar em |1⟩
2. Aplicar Hadamard em todos os qubits
3. Aplicar o oráculo
4. Aplicar Hadamard nos qubits de entrada
5. Medir

In [ ]:
def deutsch_jozsa(oracle: QuantumCircuit, n_qubits: int) -> QuantumCircuit:
    """
    Implementa o algoritmo de Deutsch-Jozsa.
    
    Args:
        oracle: o oráculo (constante ou balanceado)
        n_qubits: número de qubits de entrada
    
    Returns:
        QuantumCircuit: circuito completo
    """
    # Criar registros
    qr = QuantumRegister(n_qubits, 'input')
    ancilla = QuantumRegister(1, 'ancilla')
    cr = ClassicalRegister(n_qubits, 'output')
    
    circuit = QuantumCircuit(qr, ancilla, cr)
    
    # Passo 1: Inicializar ancilla em |1⟩
    circuit.x(ancilla[0])
    
    # Passo 2: Aplicar Hadamard em todos os qubits
    circuit.h(qr)
    circuit.h(ancilla[0])
    
    circuit.barrier()
    
    # Passo 3: Aplicar o oráculo
    circuit.compose(oracle, inplace=True)
    
    circuit.barrier()
    
    # Passo 4: Aplicar Hadamard nos qubits de entrada
    circuit.h(qr)
    
    # Passo 5: Medir os qubits de entrada
    circuit.measure(qr, cr)
    
    return circuit


# Visualizar o algoritmo com oráculo constante
oracle_const = create_constant_oracle(2, 0)
dj_const = deutsch_jozsa(oracle_const, 2)
print("Deutsch-Jozsa com oráculo CONSTANTE:")
print(dj_const.draw())

## 3. Testar com 2 Qubits

In [ ]:
# Testar com 2 Qubits - Oráculo CONSTANTE
print("=" * 50)
print("TESTE COM 2 QUBITS - ORÁCULO CONSTANTE")
print("=" * 50)

oracle_const = create_constant_oracle(2, 0)
circuit_const = deutsch_jozsa(oracle_const, 2)

# Simular
simulator = AerSimulator()
job = simulator.run(circuit_const, shots=1000)
result = job.result()
counts = result.get_counts()

print(f"\nResultados: {counts}")
print("\nInterpretação: Resultado '00' = função CONSTANTE")

# Visualizar
plot_histogram(counts)
plt.title("Deutsch-Jozsa: Oráculo Constante (2 qubits)")
plt.show()

## 4. Testar com 3 Qubits

In [ ]:
# Testar com 3 Qubits - Oráculo BALANCEADO
print("=" * 50)
print("TESTE COM 3 QUBITS - ORÁCULO BALANCEADO")
print("=" * 50)

oracle_balanced = create_balanced_oracle(3)
circuit_balanced = deutsch_jozsa(oracle_balanced, 3)

print("Circuito:")
print(circuit_balanced.draw())

# Simular
job = simulator.run(circuit_balanced, shots=1000)
result = job.result()
counts = result.get_counts()

print(f"\nResultados: {counts}")
print("\nInterpretação: Resultado != '000' = função BALANCEADA")

# Visualizar
plot_histogram(counts)
plt.title("Deutsch-Jozsa: Oráculo Balanceado (3 qubits)")
plt.show()

## 5. Análise

Responda:
- Como você identifica se o oráculo é constante ou balanceado pelo resultado da medição?
- Quantas consultas ao oráculo foram necessárias?

**Resposta:**

### Como identificar se o oráculo é constante ou balanceado?

- **Resultado = |00...0⟩**: A função é **CONSTANTE**
- **Resultado ≠ |00...0⟩**: A função é **BALANCEADA**

Isso funciona porque:
1. Se f é constante, a fase aplicada é a mesma para todos os estados, e a interferência destrutiva cancela tudo exceto |0⟩
2. Se f é balanceada, as fases se cancelam de forma diferente, resultando em um estado ortogonal a |0⟩

### Quantas consultas ao oráculo foram necessárias?

- **Algoritmo clássico**: Até 2^(n-1) + 1 consultas no pior caso
- **Deutsch-Jozsa**: **Apenas 1 consulta!**

Para n=3 qubits (8 possíveis entradas):
- Clássico: até 5 consultas
- Quântico: 1 consulta

**Speedup exponencial demonstrado!**